# Pretrained ResNet18, full training set

ResNet18 starts from **ImageNet** weights and is trained on the **full** Amazon training split, with no data augmentation.

Each image is a 256×256 RGB satellite chip and can carry several of 17 land-cover and atmosphere tags. The score is F-beta with beta = 2, the metric used in the Planet competition.

Run this notebook in Google Colab with a GPU. The image archive is downloaded in the data section below.

Code adapted from course notebooks by Indu Panigrahi and Carlos Jimenez and from [Vincent Maladiere's ResNet18 notebook](https://www.kaggle.com/code/vincentmaladiere/pytorch-resnet18-0-93). Images and labels are from Kaggle's [Planet: Understanding the Amazon from Space](https://www.kaggle.com/competitions/planet-understanding-the-amazon-from-space) competition.


## Setup


In [ ]:
import os
import pickle
import random

import cv2
import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn as nn
from plotly import graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, fbeta_score
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms as T
from tqdm.notebook import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
random.seed(101)
np.random.seed(101)
torch.manual_seed(101)


## Data

Download the full training split and the label file, then keep only the rows whose images are in this split. Tags are stored as space-separated names, so `haze primary` becomes `["haze", "primary"]`.


In [ ]:
!pip install -q gdown
!gdown https://drive.google.com/uc?id=1rD8NzCsUJbuUyH-b6zDylp8m4ZVPY9lH -O amazon-dataset-split-all.tar.gz
!gdown https://drive.google.com/uc?id=1Qpdhlky4uaMBj4kvsKnFWD1nAsEaI5UH -O train_classes.csv
!tar -xzf amazon-dataset-split-all.tar.gz && rm -f amazon-dataset-split-all.tar.gz


In [ ]:
train_paths = ["amazon-dataset-split-all/train"]
val_paths = ["amazon-dataset-split-all/validation"]

num_train = sum(len(os.listdir(path)) for path in train_paths)
num_val = sum(len(os.listdir(path)) for path in val_paths)
print(f"train files: {num_train}, validation files: {num_val}")

df_class = pd.read_csv("train_classes.csv")
df_class["list_tags"] = df_class.tags.str.split(" ")
df_class.head()


## Labels

An image can have more than one tag, so each label vector has 17 entries: 1 if that tag is present, 0 if it is not. `primary` plus `haze` is 1 at those two positions and 0 everywhere else.

`artisinal_mine` is spelled that way in the Kaggle labels.


In [ ]:
LABEL_IDXS = {
    "agriculture": 0,
    "artisinal_mine": 1,
    "bare_ground": 2,
    "blooming": 3,
    "blow_down": 4,
    "clear": 5,
    "cloudy": 6,
    "conventional_mine": 7,
    "cultivation": 8,
    "habitation": 9,
    "haze": 10,
    "partly_cloudy": 11,
    "primary": 12,
    "road": 13,
    "selective_logging": 14,
    "slash_burn": 15,
    "water": 16,
}
INVERSE_LABEL_IDXS = {index: name for name, index in LABEL_IDXS.items()}


def transform_targets(tag_list):
    targets = np.zeros(len(LABEL_IDXS), dtype=np.int8)
    targets[[LABEL_IDXS[tag] for tag in tag_list]] = 1
    return targets.tolist()


def inverse_transform_targets(targets):
    names = [INVERSE_LABEL_IDXS[index] for index, present in enumerate(targets) if present]
    return sorted(names)


## Images

ResNet18 expects spatial sizes that are multiples of 32. These chips are 256×256, so they are resized to 224×224 and normalized with the ImageNet mean and standard deviation.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_transforms():
    """Resize chips to 224 and normalize with ImageNet statistics."""
    normalize = [
        T.Resize(224),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    transform_train = T.Compose([T.ToPILImage(), *normalize])
    transform_val = T.Compose([T.ToPILImage(), *normalize])
    return transform_train, transform_val


In [ ]:
class AmazonDataset(Dataset):
    def __init__(self, df, transform, paths):
        super().__init__()
        self.df = df
        self.ohe_tags = [transform_targets(tags) for tags in df.list_tags]
        self.transform = transform
        if isinstance(paths, str):
            self.paths = [paths]
        elif isinstance(paths, (list, tuple)):
            self.paths = list(paths)
        else:
            raise ValueError("paths must be a string, list, or tuple")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.iloc[idx].image_name + ".jpg"
        matches = [
            os.path.join(path, filename)
            for path in self.paths
            if os.path.isfile(os.path.join(path, filename))
        ]
        if len(matches) != 1:
            raise FileNotFoundError(f"{filename} matched {len(matches)} files in {self.paths}")

        image = cv2.imread(matches[0])
        if image is None:
            raise ValueError(f"Could not read {matches[0]}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image, self.ohe_tags[idx]

    def collate_fn(self, batch):
        images, labels = [], []
        for image, label in batch:
            images.append(self.transform(image)[None])
            labels.append(label)
        images = torch.cat(images).float().to(device)
        labels = torch.tensor(labels).float().to(device)
        return images, labels


def load_split(paths, labels):
    if isinstance(paths, str):
        paths = [paths]
    names = set()
    for path in paths:
        names.update(name.split(".", 1)[0] for name in os.listdir(path))
    return labels[labels.image_name.isin(names)].reset_index(drop=True)


df_train = load_split(train_paths, df_class)
df_val = load_split(val_paths, df_class)

transform_train, transform_val = get_transforms()
ds_train = AmazonDataset(df_train, transform_train, paths=train_paths)
ds_val = AmazonDataset(df_val, transform_val, paths=val_paths)
print(f"train rows: {len(df_train)}, validation rows: {len(df_val)}")


## Model

ResNet18 with a new classifier head. The head maps the 512-dimensional pooled features to the 17 tags. A sigmoid on each output makes this multi-label: every tag is its own yes-or-no decision.

This run loads ImageNet weights.


In [ ]:
def get_model(pretrained=False, dropout_rate=0.2):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    model.avgpool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
    model.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate),
        nn.Linear(128, 17),
        nn.Sigmoid(),
    )
    return model.to(device)


## Training

Loss is binary cross-entropy, one term per tag. During training, a tag counts as predicted when its probability is above 0.2. That cutoff is only for the per-epoch score. The results section picks a threshold per tag afterward.

The learning rate starts at the value below and drops by 10x after epoch 7. Checkpoints go to Google Drive. The saved weights are from the last epoch. The saved predictions are from the epoch with the lowest validation loss.


In [ ]:
def train_batch(X, Y, model, loss_fn, optimizer):
    model.train()
    optimizer.zero_grad()
    Y_hat = model(X)
    batch_loss = loss_fn(Y_hat, Y)
    batch_loss.backward()
    optimizer.step()
    return Y_hat.detach().float().cpu().numpy(), batch_loss.item()


@torch.no_grad()
def compute_val_loss(X, Y, model, loss_fn):
    model.eval()
    Y_hat = model(X)
    batch_loss = loss_fn(Y_hat, Y)
    return Y_hat.float().cpu().numpy(), batch_loss.item()


In [ ]:
def train_model(model, optimizer, loss_fn, dl_train, dl_val, epochs, filename):
    lr_scheduler = StepLR(optimizer, step_size=7, gamma=0.1)
    loss_train, loss_val = [], []
    score_train, score_val = [], []
    Y_hat_val = None
    Y_thresh_val = None
    Y_val = None
    best_loss_val = np.inf

    for epoch in range(epochs):
        loss_train_epoch, loss_val_epoch = [], []
        Y_hat_train_epoch, Y_hat_val_epoch = [], []
        Y_train_epoch, Y_val_epoch = [], []

        for X, Y in tqdm(dl_train, leave=False):
            Y_hat, batch_loss = train_batch(X, Y, model, loss_fn, optimizer)
            loss_train_epoch.append(batch_loss)
            Y_hat_train_epoch.extend(Y_hat)
            Y_train_epoch.extend(Y.detach().float().cpu().numpy())

        for X, Y in tqdm(dl_val, leave=False):
            Y_hat, batch_loss = compute_val_loss(X, Y, model, loss_fn)
            loss_val_epoch.append(batch_loss)
            Y_hat_val_epoch.extend(Y_hat)
            Y_val_epoch.extend(Y.detach().float().cpu().numpy())

        Y_hat_train_epoch = np.array(Y_hat_train_epoch)
        Y_hat_val_epoch = np.array(Y_hat_val_epoch)
        Y_train_epoch = np.array(Y_train_epoch)
        Y_val_epoch = np.array(Y_val_epoch)
        Y_thresh_train_epoch = (Y_hat_train_epoch > 0.2).astype(float)
        Y_thresh_val_epoch = (Y_hat_val_epoch > 0.2).astype(float)

        avg_loss_train = np.mean(loss_train_epoch)
        avg_loss_val = np.mean(loss_val_epoch)
        score_train_epoch = fbeta_score(
            Y_train_epoch, Y_thresh_train_epoch, beta=2, average="samples"
        )
        score_val_epoch = fbeta_score(
            Y_val_epoch, Y_thresh_val_epoch, beta=2, average="samples"
        )

        if avg_loss_val < best_loss_val:
            best_loss_val = avg_loss_val
            Y_hat_val = Y_hat_val_epoch
            Y_thresh_val = Y_thresh_val_epoch
            Y_val = Y_val_epoch

        loss_train.append(avg_loss_train)
        loss_val.append(avg_loss_val)
        score_train.append(score_train_epoch)
        score_val.append(score_val_epoch)
        print(
            f"epoch: {epoch}/{epochs} -- train loss: {avg_loss_train}, "
            f"val loss: {avg_loss_val} -- train fbeta_score: {score_train_epoch}, "
            f"val fbeta_score: {score_val_epoch}"
        )
        lr_scheduler.step()

    train_results = {
        "loss_train": loss_train,
        "loss_val": loss_val,
        "score_train": score_train,
        "score_val": score_val,
        "Y_hat_val": Y_hat_val,
        "Y_thresh_val": Y_thresh_val,
        "Y_val": Y_val,
    }
    torch.save(model, filename + ".pt")
    with open(filename + ".results", "wb") as handle:
        pickle.dump(train_results, handle)
    print(f"Saved model to {filename}.pt")
    print(f"Saved results to {filename}.results")


## Rare labels

These seven tags each appear in well under 5% of images. The printout is the share of this split, not of the full Kaggle set.


In [ ]:
RARE_CLASSES = [
    "bare_ground",
    "selective_logging",
    "artisinal_mine",
    "blooming",
    "slash_burn",
    "blow_down",
    "conventional_mine",
]

for rare_class in RARE_CLASSES:
    total_train = df_train.tags.str.contains(rare_class).sum()
    total_val = df_val.tags.str.contains(rare_class).sum()
    print(
        f"train {rare_class}: {100 * total_train / len(df_train):.4f}% ({total_train})"
    )
    print(f"val {rare_class}: {100 * total_val / len(df_val):.4f}% ({total_val})\n")


In [ ]:
BATCH_SIZE = 64
LR = 1e-4
EPOCHS = 10

dl_train = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ds_train.collate_fn,
)
dl_val = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ds_val.collate_fn,
)

model = get_model(pretrained=True)
optimizer = Adam(model.parameters(), lr=LR)
loss_fn = nn.BCELoss()
filename = (
    f"/content/drive/MyDrive/pretrained_all_resnet18"
    f"__batch_size={BATCH_SIZE}__lr={LR}__epochs={EPOCHS}"
)

train_model(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    dl_train=dl_train,
    dl_val=dl_val,
    epochs=EPOCHS,
    filename=filename,
)


## Results

Loss and sample-averaged F-beta over epochs, then the average predicted probability for images that do and do not have each tag.


In [ ]:
model = torch.load(filename + ".pt", map_location=device)
with open(filename + ".results", "rb") as handle:
    train_results = pickle.load(handle)

loss_train = train_results["loss_train"]
loss_val = train_results["loss_val"]
score_train = train_results["score_train"]
score_val = train_results["score_val"]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss", "F-beta"))
fig.add_trace(go.Scatter(y=loss_train, name="train loss"), row=1, col=1)
fig.add_trace(go.Scatter(y=loss_val, name="validation loss"), row=1, col=1)
fig.add_trace(go.Scatter(y=score_train, name="train F-beta"), row=1, col=2)
fig.add_trace(go.Scatter(y=score_val, name="validation F-beta"), row=1, col=2)
fig.show()


In [ ]:
Y_hat_val = np.array(train_results["Y_hat_val"])
Y_val = np.array(train_results["Y_val"])

pos_probas, neg_probas = [], []
for _, index in LABEL_IDXS.items():
    pos_probas.append(Y_hat_val[Y_val[:, index] == 1, index].mean())
    neg_probas.append(Y_hat_val[Y_val[:, index] == 0, index].mean())

go.Figure([
    go.Bar(x=list(LABEL_IDXS), y=pos_probas, name="mean probability when the tag is present"),
    go.Bar(x=list(LABEL_IDXS), y=neg_probas, name="mean probability when the tag is absent"),
]).show()


## Thresholds and per-tag scores

The 0.2 cutoff used while training is not tuned. This search tries 100 cutoffs for each tag and keeps the one with the best sample-averaged F-beta. The bar chart then scores each tag on its own.


In [ ]:
def find_best_thresholds(Y_hat, Y, resolution=100):
    best_threshs = [0.2] * Y.shape[1]
    for tag_index in tqdm(range(Y.shape[1])):
        best_score = 0
        for step in range(resolution):
            candidate = step / resolution
            threshs = best_threshs.copy()
            threshs[tag_index] = candidate
            score = fbeta_score(Y, (Y_hat > threshs).astype(float), beta=2, average="samples")
            if score > best_score:
                best_score = score
                best_threshs[tag_index] = candidate

    overall = fbeta_score(Y, (Y_hat > best_threshs).astype(float), beta=2, average="samples")
    print(f"thresholds: {best_threshs}")
    print(f"best sample-averaged F-beta: {overall}")
    return best_threshs


threshs = find_best_thresholds(Y_hat_val, Y_val)

class_scores = {}
for index, name in INVERSE_LABEL_IDXS.items():
    y_true = Y_val[:, index].ravel()
    y_pred = (Y_hat_val[:, index].ravel() > threshs[index]).astype(float)
    class_scores[name] = round(fbeta_score(y_true, y_pred, beta=2), 4)

df_score = pd.DataFrame({
    "label": list(class_scores),
    "score": list(class_scores.values()),
}).sort_values("score", ascending=False)
px.bar(df_score, x="label", y="score", color="score", title="F-beta by label").show()


## Confusion matrices

One matrix per tag, using that tag's threshold. Counts are false negatives, true negatives, true positives, and false positives.


In [ ]:
fig = make_subplots(cols=5, rows=4)
for index, name in INVERSE_LABEL_IDXS.items():
    y_true = Y_val[:, index].ravel()
    y_pred = (Y_hat_val[:, index].ravel() > threshs[index]).astype(float)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    matrix = np.array([[fn, tn], [tp, fp]])
    row = index % 4 + 1
    col = index // 4 + 1
    fig.add_trace(
        go.Heatmap(
            z=matrix,
            text=[[f"fn: {fn}", f"tn: {tn}"], [f"tp: {tp}", f"fp: {fp}"]],
            texttemplate="%{text}",
            colorscale="Viridis",
            showscale=False,
            name=name,
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(title=name, showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)

fig.update_layout(width=1200, height=800, title="Confusion matrices")
fig.show()
